<a href="https://colab.research.google.com/github/GokhanMurali/learn2branchbyGAT/blob/main/learn2branchbyGAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Branching with Imitation Learning and a GNN

In this tutorial we will reproduce a simplified version of the paper of Gasse et al. (2019) on learning to branch with Ecole with `pytorch` and `pytorch geometric`. We collect strong branching examples on randomly generated maximum set covering instances, then train a graph neural network with bipartite state encodings to imitate the expert by classification. Finally, we will evaluate the quality of the policy.

The biggest difference with Gasse et al. (2019) is that only n=1,000 training examples of expert decisions are collected for training, to keep the time needed to run the tutorial reasonable. As a consequence, the resulting policy is undertrained and is not competitive with SCIP's default branching rule.

Users that are interested in reproducing competitive performance should use a larger sample size, such as the n=100,000 samples used for training in the paper. In this case, we strongly recommend to parallelize data collection, as in the original Gasse et al. (2019) code.

### Requirements
The requirements can be found in `conda-requirements.yaml`, lock files with pinned versions are also available
for various configurations.

In [ ]:
import gzip
import pickle
from pathlib import Path

import ecole
import numpy as np
import torch
import torch.nn.functional as F
import torch_geometric
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv
from torch_geometric.nn import to_hetero
from torch_geometric.nn import Linear
from torch_geometric.nn import LayerNorm
from torch_geometric.nn import HeteroConv

In [ ]:
DATA_MAX_SAMPLES = 1000
LEARNING_RATE = 0.001
NB_EPOCHS = 50
NB_EVAL_INSTANCES = 20

## 1. Data collection

Our first step will be to run explore-then-strong-branch on randomly generated maximum set covering instances, and save the branching decisions to build a dataset. We will also record the state of the branch-and-bound process as a bipartite graph, which is already implemented in Ecole with the same features as Gasse et al. (2019).

We will use the Ecole-provided set cover instance generator.

In [ ]:
instances = ecole.instance.SetCoverGenerator(n_rows=500, n_cols=1000, density=0.05)

The explore-then-strong-branch scheme described in the paper is not implemented by default in Ecole. In this scheme, to diversify the states in which we collect examples of strong branching behavior, we mostly follow a weak but cheap expert (pseudocost branching) and only occasionally call the strong expert (strong branching). This also ensures that samples are closer to being independent and identically distributed.

This can be realized in Ecole by creating a custom observation function, which will randomly compute and return the pseudocost scores (cheap) or the strong branching scores (expensive). It also showcases extensibility in Ecole by showing how easily a custom observation function can be created and used, directly in Python.

In [ ]:
class ExploreThenStrongBranch:
    """
    This custom observation function class will randomly return either strong branching scores (expensive expert)
    or pseudocost scores (weak expert for exploration) when called at every node.
    """

    def __init__(self, expert_probability):
        self.expert_probability = expert_probability
        self.pseudocosts_function = ecole.observation.Pseudocosts()
        self.strong_branching_function = ecole.observation.StrongBranchingScores()

    def before_reset(self, model):
        """
        This function will be called at initialization of the environment (before dynamics are reset).
        """
        self.pseudocosts_function.before_reset(model)
        self.strong_branching_function.before_reset(model)

    def extract(self, model, done):
        """
        Should we return strong branching or pseudocost scores at time node?
        """
        probabilities = [1 - self.expert_probability, self.expert_probability]
        expert_chosen = bool(np.random.choice(np.arange(2), p=probabilities))
        if expert_chosen:
            return (self.strong_branching_function.extract(model, done), True)
        else:
            return (self.pseudocosts_function.extract(model, done), False)

We can now create the environment with the correct parameters (no restarts, 1h time limit, 5% expert sampling probability).

Besides the (pseudocost or strong branching) scores, our environment will return the node bipartite graph representation of
branch-and-bound states used in Gasse et al. (2019), using the `ecole.observation.NodeBipartite` observation function.
On one side of that bipartite graph, nodes represent the variables of the problem, with a vector encoding features of
that variable. On the other side of the bipartite graph, nodes represent the constraints of the problem, similarly with
a vector encoding features of that constraint. An edge links a variable and a constraint node if the variable participates
in that constraint, that is, its coefficient is nonzero in that constraint. The constraint coefficient is attached as an
attribute of the edge.

In [ ]:
# We can pass custom SCIP parameters easily
scip_parameters = {
    "separating/maxrounds": 0,
    "presolving/maxrestarts": 0,
    "limits/time": 3600,
}

# Note how we can tuple observation functions to return complex state information
env = ecole.environment.Branching(
    observation_function=(
        ExploreThenStrongBranch(expert_probability=0.05),
        ecole.observation.NodeBipartite(),
    ),
    scip_params=scip_parameters,
)

# This will seed the environment for reproducibility
env.seed(0)

Now we loop over the instances, following the strong branching expert 5% of the time and saving its decision, until enough samples are collected.

In [ ]:
episode_counter, sample_counter = 0, 0
Path("samples/").mkdir(exist_ok=True)

# We will solve problems (run episodes) until we have saved enough samples
while sample_counter < DATA_MAX_SAMPLES:
    episode_counter += 1

    observation, action_set, _, done, _ = env.reset(next(instances))
    while not done:
        (scores, scores_are_expert), node_observation = observation
        action = action_set[scores[action_set].argmax()]

        # Only save samples if they are coming from the expert (strong branching)
        if scores_are_expert and (sample_counter < DATA_MAX_SAMPLES):
            sample_counter += 1
            data = [node_observation, action, action_set, scores]
            filename = f"samples/sample_{sample_counter}.pkl"

            constraint_features = node_observation.row_features
            edge_indices = node_observation.edge_features.indices.astype(np.int32)
            edge_features = np.expand_dims(node_observation.edge_features.values, axis=-1)
            variable_features = node_observation.variable_features

            with gzip.open(filename, "wb") as f:
                pickle.dump(data, f)

        observation, action_set, _, done, _ = env.step(action)

    print(f"Episode {episode_counter}, {sample_counter} samples collected so far")

Episode 1, 2 samples collected so far
Episode 2, 13 samples collected so far
Episode 3, 14 samples collected so far
Episode 4, 20 samples collected so far
Episode 5, 20 samples collected so far
Episode 6, 23 samples collected so far
Episode 7, 33 samples collected so far
Episode 8, 34 samples collected so far
Episode 9, 35 samples collected so far
Episode 10, 55 samples collected so far
Episode 11, 59 samples collected so far
Episode 12, 67 samples collected so far
Episode 13, 68 samples collected so far
Episode 14, 70 samples collected so far
Episode 15, 73 samples collected so far
Episode 16, 73 samples collected so far
Episode 17, 78 samples collected so far
Episode 18, 86 samples collected so far
Episode 19, 95 samples collected so far
Episode 20, 95 samples collected so far
Episode 21, 96 samples collected so far
Episode 22, 99 samples collected so far
Episode 23, 104 samples collected so far
Episode 24, 114 samples collected so far
Episode 25, 119 samples collected so far
Episode

# 2. Train a GNN

Our next step is to train a GNN classifier on these collected samples to predict similar choices to strong branching.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

We will first define pytorch geometric data classes to handle the bipartite graph data.

In [ ]:
class BipartiteNodeData(torch_geometric.data.Data):
    """
    This class encode a node bipartite graph observation as returned by the `ecole.observation.NodeBipartite`
    observation function in a format understood by the pytorch geometric data handlers.
    """

    def __init__(
        self,
        constraint_features,
        edge_indices,
        edge_features,
        variable_features,
        candidates,
        nb_candidates,
        candidate_choice,
        candidate_scores,
    ):
        super().__init__()
        self.constraint_features = constraint_features
        self.edge_index = edge_indices
        self.edge_attr = edge_features
        self.variable_features = variable_features
        self.candidates = candidates
        self.nb_candidates = nb_candidates
        self.candidate_choices = candidate_choice
        self.candidate_scores = candidate_scores

    def __inc__(self, key, value, store, *args, **kwargs):
        """
        We overload the pytorch geometric method that tells how to increment indices when concatenating graphs
        for those entries (edge index, candidates) for which this is not obvious.
        """
        if key == "edge_index":
            return torch.tensor(
                [[self.constraint_features.size(0)], [self.variable_features.size(0)]]
            )
        elif key == "candidates":
            return self.variable_features.size(0)
        else:
            return super().__inc__(key, value, *args, **kwargs)


class GraphDataset(torch_geometric.data.Dataset):
    """
    This class encodes a collection of graphs, as well as a method to load such graphs from the disk.
    It can be used in turn by the data loaders provided by pytorch geometric.
    """

    def __init__(self, sample_files):
        super().__init__(root=None, transform=None, pre_transform=None)
        self.sample_files = sample_files

    def len(self):
        return len(self.sample_files)

    def get(self, index):
        """
        This method loads a node bipartite graph observation as saved on the disk during data collection.
        """
        with gzip.open(self.sample_files[index], "rb") as f:
            sample = pickle.load(f)

        sample_observation, sample_action, sample_action_set, sample_scores = sample

        constraint_features = sample_observation.row_features
        edge_indices = sample_observation.edge_features.indices.astype(np.int32)
        edge_features = np.expand_dims(sample_observation.edge_features.values, axis=-1)
        variable_features = sample_observation.variable_features

        # We note on which variables we were allowed to branch, the scores as well as the choice
        # taken by strong branching (relative to the candidates)
        candidates = np.array(sample_action_set, dtype=np.int32)
        candidate_scores = np.array([sample_scores[j] for j in candidates])
        candidate_choice = np.where(candidates == sample_action)[0][0]

        graph = BipartiteNodeData(
            torch.FloatTensor(constraint_features),
            torch.LongTensor(edge_indices),
            torch.FloatTensor(edge_features),
            torch.FloatTensor(variable_features),
            torch.LongTensor(candidates),
            len(candidates),
            torch.LongTensor([candidate_choice]),
            torch.FloatTensor(candidate_scores)
        )

        # We must tell pytorch geometric how many nodes there are, for indexing purposes
        graph.num_nodes = constraint_features.shape[0] + variable_features.shape[0]

        return graph

In [ ]:
class HeteroGraphDataset(torch_geometric.data.Dataset):
    """
    This class encodes a collection of graphs, as well as a method to load such graphs from the disk.
    It can be used in turn by the data loaders provided by pytorch geometric.
    """

    def __init__(self, sample_files):
        super().__init__(root=None, transform=None, pre_transform=None)
        self.sample_files = sample_files

    def len(self):
        return len(self.sample_files)

    def get(self, index):
        """
        This method loads a node bipartite graph observation as saved on the disk during data collection.
        """
        with gzip.open(self.sample_files[index], "rb") as f:
            sample = pickle.load(f)

        sample_observation, sample_action, sample_action_set, sample_scores = sample

        graph = HeteroData()

        #node features
        graph['constraint'].x = torch.FloatTensor(sample_observation.row_features)
        graph['variable'].x = torch.FloatTensor(sample_observation.variable_features)

        #edge indices
        graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(sample_observation.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(sample_observation.edge_features.indices.astype(np.int32)[0])], dim=0)
        graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(sample_observation.edge_features.indices.astype(np.int32))

        #edge features
        graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(sample_observation.edge_features.values, axis=-1))
        graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(sample_observation.edge_features.values, axis=-1))

        #graph features
        graph['candidates'] = torch.LongTensor(np.array(sample_action_set, dtype=np.int32))
        graph['candidate_scores'] = torch.FloatTensor(np.array([sample_scores[j] for j in np.array(sample_action_set, dtype=np.int32)]))
        graph['candidate_choices'] = torch.LongTensor([np.where(np.array(sample_action_set, dtype=np.int32) == sample_action)[0][0]])
        graph['nb_candidates'] = len(np.array(sample_action_set, dtype=np.int32))

        return graph

We can then prepare the data loaders.

In [ ]:
sample_files = [str(path) for path in Path("samples/").glob("sample_*.pkl")]
train_files = sample_files[: int(0.8 * len(sample_files))]
valid_files = sample_files[int(0.8 * len(sample_files)) :]

train_data = GraphDataset(train_files)
train_loader = torch_geometric.loader.DataLoader(train_data, batch_size=32, shuffle=True)
valid_data = GraphDataset(valid_files)
valid_loader = torch_geometric.loader.DataLoader(valid_data, batch_size=128, shuffle=False)

In [ ]:
hetero_train_data = HeteroGraphDataset(train_files)
hetero_valid_data = HeteroGraphDataset(valid_files)

Next, we will define our graph neural network architecture.

In [ ]:
class GNNPolicy(torch.nn.Module):
    def __init__(self):
        super().__init__()
        emb_size = 64
        cons_nfeats = 5
        edge_nfeats = 1
        var_nfeats = 19

        # CONSTRAINT EMBEDDING
        self.cons_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(cons_nfeats),
            torch.nn.Linear(cons_nfeats, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
        )

        # EDGE EMBEDDING
        self.edge_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(edge_nfeats),
        )

        # VARIABLE EMBEDDING
        self.var_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(var_nfeats),
            torch.nn.Linear(var_nfeats, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
        )

        self.conv_v_to_c = BipartiteGraphConvolution()
        self.conv_c_to_v = BipartiteGraphConvolution()

        self.output_module = torch.nn.Sequential(
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, 1, bias=False),
        )

    def forward(
        self, constraint_features, edge_indices, edge_features, variable_features
    ):
        reversed_edge_indices = torch.stack([edge_indices[1], edge_indices[0]], dim=0)

        # First step: linear embedding layers to a common dimension (64)
        constraint_features = self.cons_embedding(constraint_features)
        edge_features = self.edge_embedding(edge_features)
        variable_features = self.var_embedding(variable_features)

        # Two half convolutions
        constraint_features = self.conv_v_to_c(
            variable_features, reversed_edge_indices, edge_features, constraint_features
        )
        variable_features = self.conv_c_to_v(
            constraint_features, edge_indices, edge_features, variable_features
        )

        # A final MLP on the variable features
        output = self.output_module(variable_features).squeeze(-1)
        return output


class BipartiteGraphConvolution(torch_geometric.nn.MessagePassing):
    """
    The bipartite graph convolution is already provided by pytorch geometric and we merely need
    to provide the exact form of the messages being passed.
    """

    def __init__(self):
        super().__init__("add")
        emb_size = 64

        self.feature_module_left = torch.nn.Sequential(
            torch.nn.Linear(emb_size, emb_size)
        )
        self.feature_module_edge = torch.nn.Sequential(
            torch.nn.Linear(1, emb_size, bias=False)
        )
        self.feature_module_right = torch.nn.Sequential(
            torch.nn.Linear(emb_size, emb_size, bias=False)
        )
        self.feature_module_final = torch.nn.Sequential(
            torch.nn.LayerNorm(emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
        )

        self.post_conv_module = torch.nn.Sequential(torch.nn.LayerNorm(emb_size))

        # output_layers
        self.output_module = torch.nn.Sequential(
            torch.nn.Linear(2 * emb_size, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
        )

    def forward(self, left_features, edge_indices, edge_features, right_features):
        """
        This method sends the messages, computed in the message method.
        """
        output = self.propagate(
            edge_indices,
            size=(left_features.shape[0], right_features.shape[0]),
            node_features=(left_features, right_features),
            edge_features=edge_features,
        )
        return self.output_module(
            torch.cat([self.post_conv_module(output), right_features], dim=-1)
        )

    def message(self, node_features_i, node_features_j, edge_features):
        output = self.feature_module_final(
            self.feature_module_left(node_features_i)
            + self.feature_module_edge(edge_features)
            + self.feature_module_right(node_features_j)
        )
        return output


policy = GNNPolicy().to(DEVICE)

In [ ]:
class HeteroGATPolicy(torch.nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        emb_size = 64
        cons_nfeats = 5
        edge_nfeats = 1
        var_nfeats = 19

        # CONSTRAINT EMBEDDING
        self.cons_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(cons_nfeats),
            torch.nn.Linear(cons_nfeats, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
        )

        # EDGE EMBEDDING
        self.edge_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(edge_nfeats),
        )

        # VARIABLE EMBEDDING
        self.var_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(var_nfeats),
            torch.nn.Linear(var_nfeats, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
        )

        self.conv1 = HeteroConv({
                ('variable', 'participates', 'constraint'): GATConv((-1, -1), emb_size, heads=num_heads, add_self_loops=False),
                ('constraint', 'contains', 'variable'): GATConv((-1, -1), emb_size, heads=num_heads, add_self_loops=False),
            }, aggr='sum')

        self.lin1 = torch.nn.Linear(num_heads*emb_size, emb_size)

        self.conv2 = HeteroConv({
                ('variable', 'participates', 'constraint'): GATConv((-1, -1), emb_size, heads=1, add_self_loops=False),
                ('constraint', 'contains', 'variable'): GATConv((-1, -1), emb_size, heads=1, add_self_loops=False),
            }, aggr='sum')

        self.lin2 = torch.nn.Linear(emb_size, emb_size)

        self.output_module = torch.nn.Sequential(
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, 1, bias=False),
        )


    def forward(self, x_dict, edge_index_dict, edge_attr_dict):

        x_dict['constraint'] = self.cons_embedding(x_dict['constraint'])
        x_dict['variable'] = self.var_embedding(x_dict['variable'])
        edge_attr_dict['variable', 'participates', 'constraint'] = self.edge_embedding(edge_attr_dict['variable', 'participates', 'constraint'])
        edge_attr_dict['constraint', 'contains', 'variable'] = self.edge_embedding(edge_attr_dict['constraint', 'contains', 'variable'])

        #Layer 1
        x_dict = self.conv1(x_dict, edge_index_dict, edge_attr_dict)
        x_dict['constraint'] = self.lin1(x_dict['constraint'])
        x_dict['variable'] = self.lin1(x_dict['variable'])
        x_dict['constraint'] = x_dict['constraint'].relu()
        x_dict['variable'] = x_dict['variable'].relu()

        #Layer 2
        x_dict = self.conv2(x_dict, edge_index_dict, edge_attr_dict)
        x_dict['constraint'] = self.lin2(x_dict['constraint'])
        x_dict['variable'] = self.lin2(x_dict['variable'])
        x_dict['constraint'] = x_dict['constraint'].relu()
        x_dict['variable'] = x_dict['variable'].relu()

        #A final MLP on the variable features
        output = self.output_module(x_dict['variable']).squeeze(-1)

        return output

model = HeteroGATPolicy(num_heads=1).to(DEVICE)
model_multi_head = HeteroGATPolicy(num_heads=8).to(DEVICE)

In [ ]:
hetero_observation = hetero_train_data[0].to(DEVICE)

hetero_logits = model(hetero_observation.x_dict, hetero_observation.edge_index_dict, hetero_observation.edge_attr_dict)

action_distribution_gat = F.softmax(hetero_logits[hetero_observation.candidates], dim=-1)

print(action_distribution_gat)

tensor([0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118,
        0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118,
        0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118,
        0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118,
        0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118,
        0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118,
        0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118,
        0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118,
        0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118,
        0.0118, 0.0118, 0.0118, 0.0118], grad_fn=<SoftmaxBackward0>)


With this model we can predict a probability distribution over actions as follows.

In [ ]:
observation = train_data[0].to(DEVICE)

logits = policy(
    observation.constraint_features,
    observation.edge_index,
    observation.edge_attr,
    observation.variable_features,
)
action_distribution = F.softmax(logits[observation.candidates], dim=-1)

print(action_distribution)

tensor([0.0118, 0.0118, 0.0118, 0.0116, 0.0118, 0.0117, 0.0116, 0.0118, 0.0117,
        0.0118, 0.0117, 0.0116, 0.0118, 0.0117, 0.0120, 0.0118, 0.0119, 0.0116,
        0.0117, 0.0118, 0.0117, 0.0118, 0.0118, 0.0118, 0.0117, 0.0118, 0.0118,
        0.0118, 0.0118, 0.0116, 0.0117, 0.0117, 0.0119, 0.0118, 0.0118, 0.0118,
        0.0118, 0.0118, 0.0118, 0.0117, 0.0118, 0.0116, 0.0118, 0.0118, 0.0117,
        0.0117, 0.0117, 0.0118, 0.0117, 0.0117, 0.0119, 0.0119, 0.0118, 0.0118,
        0.0117, 0.0117, 0.0118, 0.0118, 0.0117, 0.0119, 0.0118, 0.0117, 0.0118,
        0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0118, 0.0117, 0.0117, 0.0117,
        0.0117, 0.0118, 0.0117, 0.0118, 0.0117, 0.0118, 0.0117, 0.0117, 0.0118,
        0.0118, 0.0118, 0.0116, 0.0117], grad_fn=<SoftmaxBackward0>)


As can be seen, with randomly initialized weights, the initial distributions tend to be close to uniform.
Next, we will define two helper functions: one to train or evaluate the model on a whole epoch and compute metrics for monitoring, and one for padding tensors when doing predictions on a batch of graphs of potentially different number of variables.

In [ ]:
def process(policy, data_loader, optimizer=None):
    """
    This function will process a whole epoch of training or validation, depending on whether an optimizer is provided.
    """
    mean_loss = 0
    mean_acc = 0

    n_samples_processed = 0
    with torch.set_grad_enabled(optimizer is not None):
        for batch in data_loader:
            batch = batch.to(DEVICE)
            # Compute the logits (i.e. pre-softmax activations) according to the policy on the concatenated graphs
            logits = policy(
                batch.constraint_features,
                batch.edge_index,
                batch.edge_attr,
                batch.variable_features,
            )
            # Index the results by the candidates, and split and pad them
            logits = pad_tensor(logits[batch.candidates], batch.nb_candidates)
            # Compute the usual cross-entropy classification loss
            loss = F.cross_entropy(logits, batch.candidate_choices)

            if optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            true_scores = pad_tensor(batch.candidate_scores, batch.nb_candidates)
            true_bestscore = true_scores.max(dim=-1, keepdims=True).values

            predicted_bestindex = logits.max(dim=-1, keepdims=True).indices
            accuracy = (
                (true_scores.gather(-1, predicted_bestindex) == true_bestscore)
                .float()
                .mean()
                .item()
            )

            mean_loss += loss.item() * batch.num_graphs
            mean_acc += accuracy * batch.num_graphs
            n_samples_processed += batch.num_graphs

    mean_loss /= n_samples_processed
    mean_acc /= n_samples_processed
    return mean_loss, mean_acc


def pad_tensor(input_, pad_sizes, pad_value=-1e8):
    """
    This utility function splits a tensor and pads each split to make them all the same size, then stacks them.
    """
    max_pad_size = pad_sizes.max()
    output = input_.split(pad_sizes.cpu().numpy().tolist())
    output = torch.stack(
        [
            F.pad(slice_, (0, max_pad_size - slice_.size(0)), "constant", pad_value)
            for slice_ in output
        ],
        dim=0,
    )
    return output

In [ ]:
def gat_process(model, dataset, optimizer=None):
    """
    This function will process a whole epoch of training or validation, depending on whether an optimizer is provided.
    """
    mean_loss = 0
    mean_acc = 0

    n_samples_processed = 0
    with torch.set_grad_enabled(optimizer is not None):
        for data in dataset:
            data = data.to(DEVICE)
            # Compute the logits (i.e. pre-softmax activations) according to the policy on the concatenated graphs
            logits = model(
                data.x_dict,
                data.edge_index_dict,
                data.edge_attr_dict,
               )

            # Compute the usual cross-entropy classification loss
            loss = F.cross_entropy(torch.reshape(logits[data.candidates], (1,-1)), data.candidate_choices)

            if optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            true_scores = data.candidate_scores
            true_bestscore = true_scores.max(dim=-1, keepdims=True).values

            predicted_bestindex = logits[data.candidates].max(dim=-1, keepdims=True).indices

            mean_loss += loss.item()
            mean_acc += int(true_scores[predicted_bestindex] == true_bestscore)
            n_samples_processed += 1

    mean_loss /= n_samples_processed
    mean_acc /= n_samples_processed
    return mean_loss, mean_acc


After this, we can actually create the model and train it.

In [ ]:
optimizer = torch.optim.Adam(policy.parameters(), lr=LEARNING_RATE)
for epoch in range(NB_EPOCHS):
    print(f"Epoch {epoch+1}")

    train_loss, train_acc = process(policy, train_loader, optimizer)
    print(f"Train loss: {train_loss:0.3f}, accuracy {train_acc:0.3f}")

    valid_loss, valid_acc = process(policy, valid_loader, None)
    print(f"Valid loss: {valid_loss:0.3f}, accuracy {valid_acc:0.3f}")

torch.save(policy.state_dict(), "trained_params.pkl")

Epoch 1
Train loss: 4.298, accuracy 0.255
Valid loss: 3.969, accuracy 0.355
Epoch 2
Train loss: 3.724, accuracy 0.410
Valid loss: 3.616, accuracy 0.435
Epoch 3
Train loss: 3.541, accuracy 0.469
Valid loss: 3.485, accuracy 0.435
Epoch 4
Train loss: 3.524, accuracy 0.472
Valid loss: 3.452, accuracy 0.440
Epoch 5
Train loss: 3.487, accuracy 0.461
Valid loss: 3.501, accuracy 0.430
Epoch 6
Train loss: 3.483, accuracy 0.471
Valid loss: 3.460, accuracy 0.470
Epoch 7
Train loss: 3.490, accuracy 0.475
Valid loss: 3.430, accuracy 0.435
Epoch 8
Train loss: 3.484, accuracy 0.466
Valid loss: 3.426, accuracy 0.425
Epoch 9
Train loss: 3.468, accuracy 0.468
Valid loss: 3.477, accuracy 0.415
Epoch 10
Train loss: 3.458, accuracy 0.464
Valid loss: 3.409, accuracy 0.445
Epoch 11
Train loss: 3.436, accuracy 0.458
Valid loss: 3.414, accuracy 0.445
Epoch 12
Train loss: 3.454, accuracy 0.474
Valid loss: 3.480, accuracy 0.415
Epoch 13
Train loss: 3.447, accuracy 0.468
Valid loss: 3.400, accuracy 0.425
Epoch 14

In [ ]:
gat_optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
for epoch in range(NB_EPOCHS):
    print(f"Epoch {epoch+1}")

    gat_train_loss, gat_train_acc = gat_process(model, hetero_train_data, gat_optimizer)
    print(f"Train loss: {gat_train_loss:0.3f}, accuracy {gat_train_acc:0.3f}")

    gat_valid_loss, gat_valid_acc = gat_process(model, hetero_valid_data, None)
    print(f"Valid loss: {gat_valid_loss:0.3f}, accuracy {gat_valid_acc:0.3f}")

torch.save(model.state_dict(), "gat_trained_params.pkl")

Epoch 1
Train loss: 3.758, accuracy 0.398
Valid loss: 3.797, accuracy 0.380
Epoch 2
Train loss: 3.736, accuracy 0.399
Valid loss: 3.786, accuracy 0.390
Epoch 3
Train loss: 3.731, accuracy 0.391
Valid loss: 3.792, accuracy 0.390
Epoch 4
Train loss: 3.727, accuracy 0.385
Valid loss: 3.891, accuracy 0.385
Epoch 5
Train loss: 3.764, accuracy 0.390
Valid loss: 3.774, accuracy 0.385
Epoch 6
Train loss: 3.725, accuracy 0.393
Valid loss: 3.764, accuracy 0.390
Epoch 7
Train loss: 3.711, accuracy 0.398
Valid loss: 3.775, accuracy 0.390
Epoch 8
Train loss: 3.707, accuracy 0.404
Valid loss: 3.765, accuracy 0.400
Epoch 9
Train loss: 3.694, accuracy 0.407
Valid loss: 3.757, accuracy 0.415
Epoch 10
Train loss: 3.684, accuracy 0.405
Valid loss: 3.755, accuracy 0.420
Epoch 11
Train loss: 3.675, accuracy 0.405
Valid loss: 3.749, accuracy 0.410
Epoch 12
Train loss: 3.667, accuracy 0.410
Valid loss: 3.742, accuracy 0.415
Epoch 13
Train loss: 3.691, accuracy 0.405
Valid loss: 3.737, accuracy 0.395
Epoch 14

In [ ]:
mh_gat_optimizer = torch.optim.Adam(model_multi_head.parameters(), lr=LEARNING_RATE)
for epoch in range(NB_EPOCHS):
    print(f"Epoch {epoch+1}")

    mh_gat_train_loss, mh_gat_train_acc = gat_process(model_multi_head, hetero_train_data, mh_gat_optimizer)
    print(f"Train loss: {mh_gat_train_loss:0.3f}, accuracy {mh_gat_train_acc:0.3f}")

    mh_gat_valid_loss, mh_gat_valid_acc = gat_process(model_multi_head, hetero_valid_data, None)
    print(f"Valid loss: {mh_gat_valid_loss:0.3f}, accuracy {mh_gat_valid_acc:0.3f}")

torch.save(model_multi_head.state_dict(), "mh_gat_trained_params.pkl")

Epoch 1
Train loss: 4.093, accuracy 0.301
Valid loss: 3.793, accuracy 0.385
Epoch 2
Train loss: 3.842, accuracy 0.379
Valid loss: 3.729, accuracy 0.405
Epoch 3
Train loss: 3.769, accuracy 0.395
Valid loss: 3.699, accuracy 0.440
Epoch 4
Train loss: 3.729, accuracy 0.400
Valid loss: 3.690, accuracy 0.435
Epoch 5
Train loss: 3.710, accuracy 0.403
Valid loss: 3.688, accuracy 0.405
Epoch 6
Train loss: 3.694, accuracy 0.411
Valid loss: 3.693, accuracy 0.420
Epoch 7
Train loss: 3.689, accuracy 0.417
Valid loss: 3.696, accuracy 0.435
Epoch 8
Train loss: 3.678, accuracy 0.425
Valid loss: 3.736, accuracy 0.400
Epoch 9
Train loss: 3.671, accuracy 0.424
Valid loss: 3.690, accuracy 0.425
Epoch 10
Train loss: 3.671, accuracy 0.424
Valid loss: 3.722, accuracy 0.390
Epoch 11
Train loss: 3.655, accuracy 0.426
Valid loss: 3.698, accuracy 0.395
Epoch 12
Train loss: 3.643, accuracy 0.430
Valid loss: 3.694, accuracy 0.395
Epoch 13
Train loss: 3.646, accuracy 0.429
Valid loss: 3.676, accuracy 0.425
Epoch 14

# 3 Evaluation

Finally, we can evaluate the performance of the model. We first define appropriate environments. For benchmarking purposes, we include a trivial environment that merely runs SCIP.

In [ ]:
scip_parameters = {
    "separating/maxrounds": 0,
    "presolving/maxrestarts": 0,
    "limits/time": 3600,
}
env = ecole.environment.Branching(
    observation_function=ecole.observation.NodeBipartite(),
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
        "time": ecole.reward.SolvingTime(),
    },
    scip_params=scip_parameters,
)
gat_env = ecole.environment.Branching(
    observation_function=ecole.observation.NodeBipartite(),
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
        "time": ecole.reward.SolvingTime(),
    },
    scip_params=scip_parameters,
)
mh_gat_env = ecole.environment.Branching(
    observation_function=ecole.observation.NodeBipartite(),
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
        "time": ecole.reward.SolvingTime(),
    },
    scip_params=scip_parameters,
)
default_env = ecole.environment.Configuring(
    observation_function=None,
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
        "time": ecole.reward.SolvingTime(),
    },
    scip_params=scip_parameters,
)

Then we can simply follow the environments, taking steps appropriately according to the GNN policy.

In [ ]:
instances = ecole.instance.SetCoverGenerator(n_rows=500, n_cols=1000, density=0.05)
for instance_count, instance in zip(range(NB_EVAL_INSTANCES), instances):
    # Run the GNN brancher
    nb_nodes, time = 0, 0
    observation, action_set, _, done, info = env.reset(instance)
    nb_nodes += info["nb_nodes"]
    time += info["time"]
    while not done:
        with torch.no_grad():
            observation = (
                torch.from_numpy(observation.row_features.astype(np.float32)).to(DEVICE),
                torch.from_numpy(observation.edge_features.indices.astype(np.int64)).to(DEVICE),
                torch.from_numpy(observation.edge_features.values.astype(np.float32)).view(-1, 1).to(DEVICE),
                torch.from_numpy(observation.variable_features.astype(np.float32)).to(DEVICE),
            )
            logits = policy(*observation)
            action = action_set[logits[action_set.astype(np.int64)].argmax()]
            observation, action_set, _, done, info = env.step(action)
        nb_nodes += info["nb_nodes"]
        time += info["time"]

    # Run the GAT brancher
    gat_nb_nodes, gat_time = 0, 0
    gat_observation, gat_action_set, _, gat_done, gat_info = gat_env.reset(instance)
    gat_nb_nodes += gat_info["nb_nodes"]
    gat_time += gat_info["time"]
    while not gat_done:
        with torch.no_grad():
            graph = HeteroData()
            #node features
            graph['constraint'].x = torch.FloatTensor(gat_observation.row_features)
            graph['variable'].x = torch.FloatTensor(gat_observation.variable_features)
            #edge indices
            graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(gat_observation.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(gat_observation.edge_features.indices.astype(np.int32)[0])], dim=0)
            graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(gat_observation.edge_features.indices.astype(np.int32))
            #edge features
            graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(gat_observation.edge_features.values, axis=-1))
            graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(gat_observation.edge_features.values, axis=-1))

            gat_logits = model(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
            gat_action = gat_action_set[gat_logits[gat_action_set.astype(np.int64)].argmax()]
            gat_observation, gat_action_set, _, gat_done, gat_info = gat_env.step(gat_action)
        gat_nb_nodes += gat_info["nb_nodes"]
        gat_time += gat_info["time"]

    # Run the Multi Head GAT brancher
    mh_gat_nb_nodes, mh_gat_time = 0, 0
    mh_gat_observation, mh_gat_action_set, _, mh_gat_done, mh_gat_info = mh_gat_env.reset(instance)
    mh_gat_nb_nodes += mh_gat_info["nb_nodes"]
    mh_gat_time += mh_gat_info["time"]
    while not mh_gat_done:
        with torch.no_grad():
            graph = HeteroData()
            #node features
            graph['constraint'].x = torch.FloatTensor(mh_gat_observation.row_features)
            graph['variable'].x = torch.FloatTensor(mh_gat_observation.variable_features)
            #edge indices
            graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(mh_gat_observation.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(mh_gat_observation.edge_features.indices.astype(np.int32)[0])], dim=0)
            graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(mh_gat_observation.edge_features.indices.astype(np.int32))
            #edge features
            graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation.edge_features.values, axis=-1))
            graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation.edge_features.values, axis=-1))

            mh_gat_logits = model_multi_head(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
            mh_gat_action = mh_gat_action_set[mh_gat_logits[mh_gat_action_set.astype(np.int64)].argmax()]
            mh_gat_observation, mh_gat_action_set, _, mh_gat_done, mh_gat_info = mh_gat_env.step(mh_gat_action)
        mh_gat_nb_nodes += mh_gat_info["nb_nodes"]
        mh_gat_time += mh_gat_info["time"]

    # Run SCIP's default brancher
    default_env.reset(instance)
    _, _, _, _, default_info = default_env.step({})

    print(f"Instance {instance_count: >3} | SCIP nb nodes    {int(default_info['nb_nodes']): >4d}  | SCIP time   {default_info['time']: >6.2f} ")
    print(f"             | GNN  nb nodes    {int(nb_nodes): >4d}  | GNN  time   {time: >6.2f} ")
    print(f"             | GAT  nb nodes    {int(gat_nb_nodes): >4d}  | GAT  time   {gat_time: >6.2f} ")
    print(f"             | MH GAT  nb nodes {int(mh_gat_nb_nodes): >4d}  | MH GAT  time   {mh_gat_time: >6.2f} ")
    print(f"             | Gain1        {100*(1-nb_nodes/default_info['nb_nodes']): >8.2f}% | Gain1     {100*(1-time/default_info['time']): >8.2f}%")
    print(f"             | Gain2        {100*(1-gat_nb_nodes/nb_nodes): >8.2f}% | Gain2     {100*(1-gat_time/time): >8.2f}%")
    print(f"             | Gain3        {100*(1-mh_gat_nb_nodes/nb_nodes): >8.2f}% | Gain3     {100*(1-mh_gat_time/time): >8.2f}%")

Instance   0 | SCIP nb nodes       9  | SCIP time     5.08 
             | GNN  nb nodes      93  | GNN  time    10.13 
             | GAT  nb nodes    7443  | GAT  time   467.64 
             | MH GAT  nb nodes  121  | MH GAT  time    24.95 
             | Gain1         -933.33% | Gain1       -99.26%
             | Gain2        -7903.23% | Gain2     -4516.74%
             | Gain3          -30.11% | Gain3      -146.27%
Instance   1 | SCIP nb nodes      65  | SCIP time    10.87 
             | GNN  nb nodes     249  | GNN  time    21.90 
             | GAT  nb nodes    3621  | GAT  time   235.03 
             | MH GAT  nb nodes  237  | MH GAT  time    47.56 
             | Gain1         -283.08% | Gain1      -101.57%
             | Gain2        -1354.22% | Gain2      -973.13%
             | Gain3            4.82% | Gain3      -117.16%
Instance   2 | SCIP nb nodes      11  | SCIP time     6.90 
             | GNN  nb nodes      99  | GNN  time    10.89 
             | GAT  nb nodes    32

### References

Gasse, M., Chételat, D., Ferroni, N., Charlin, L. and Lodi, A. (2019). Exact combinatorial optimization with graph convolutional neural networks. In Advances in Neural Information Processing Systems (pp. 15580-15592).